<a href="https://colab.research.google.com/github/gregorykeune/map_reducer_clash_royale/blob/main/map_reduce_cr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install mrjob


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 6.3 MB/s eta 0:00:00


In [47]:
# Criar o script do MapReduce
%%file contar_cartas.py
# contar_cartas.py
import os
import shutil
from mrjob.job import MRJob

class MRContarCartas(MRJob):
  def mapper(self, _, line):
      # Ignorar cabeçalho (linha que começa com "rank")
      if line.startswith("rank"):
          return

      # O separador é TAB
      colunas = line.strip().split(",")

      # Ignorar as 3 primeiras colunas (rank, playerTag, name)
      for carta in colunas[3:]:
          carta = carta.strip()
          if carta:
              yield carta, 1

  def reducer(self, carta, counts):
      yield carta, sum(counts)

if __name__ == "__main__":
    # limpar diretório de saída antes de rodar
    if os.path.exists("output_cartas"):
        shutil.rmtree("output_cartas")
    MRContarCartas.run()


Overwriting contar_cartas.py


In [48]:
!python contar_cartas.py decks_top200.csv --output cartas_mais_usadas

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 1...
Creating temp directory /tmp/contar_cartas.root.20250926.231615.516776
job output is in cartas_mais_usadas
Removing temp directory /tmp/contar_cartas.root.20250926.231615.516776...


In [34]:
!python contar_cartas.py decks_top200.csv > cartas_mais_usadas/part-00000

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/contar_cartas.root.20250926.230445.609017
Running step 1 of 1...
job output is in /tmp/contar_cartas.root.20250926.230445.609017/output
Streaming final output from /tmp/contar_cartas.root.20250926.230445.609017/output...
Removing temp directory /tmp/contar_cartas.root.20250926.230445.609017...


In [51]:
%%file raridade.py
# raridade.py
import os
import shutil
from mrjob.job import MRJob

class MRRaridade(MRJob):
  def mapper(self, _, line):
    # Ignorar cabeçalho (linha que começa com "id")
    if line.startswith("id"):
        return

    # O separador é vírgula (ajuste para "\t" se for TAB)
    colunas = line.strip().split(",")

    # Pega a 3ª coluna (índice 2) -> raridade
    raridade = colunas[2].strip()
    if raridade:
      yield raridade, 1

  def reducer(self, raridade, counts):
    yield raridade, sum(counts)

if __name__ == "__main__":
    # limpar diretório de saída antes de rodar
    if os.path.exists("raridade"):
        shutil.rmtree("raridade")
    MRRaridade.run()


Writing raridade.py


In [52]:
!python raridade.py cards.csv --output raridade

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 1...
Creating temp directory /tmp/raridade.root.20250926.231738.503439
job output is in raridade
Removing temp directory /tmp/raridade.root.20250926.231738.503439...


In [87]:
%%file custo_de_elixir.py
# custo_de_elixir.py
import os
import shutil
from mrjob.job import MRJob

class MRCustoElixir(MRJob):
  def mapper(self, _, line):
    # Ignorar cabeçalho
    if line.startswith("id"):
      return

    # O separador é vírgula (use "\t" se for TAB)
    colunas = line.strip().split(",")

    # Pega a coluna 6 (índice 5) -> custo de elixir
    try:
      custo = float(colunas[4].strip())
      yield "custo", custo
    except ValueError:
      pass  # ignora se não for número

  def reducer_init(self):
    self.valores = []

  def reducer(self, key, valores):
    for v in valores:
      self.valores.append(v)

  def reducer_final(self):
    if self.valores:
      media = sum(self.valores) / len(self.valores)
      # primeiro yield: a média
      yield 'media:', round(media, 2)

      # depois: os valores individuais (contagem de cada custo)
      from collections import Counter
      contagem = Counter(self.valores)
      for custo, qtd in sorted(contagem.items()):
        yield f"custo = {custo}:", f"quantidade = {qtd}"

if __name__ == "__main__":
  if os.path.exists("avg_elixir"):
    shutil.rmtree("avg_elixir")
  MRCustoElixir.run()


Overwriting custo_de_elixir.py


In [88]:
!python custo_de_elixir.py cards.csv --output avg_elixir

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 1...
Creating temp directory /tmp/custo_de_elixir.root.20250927.001030.498089
job output is in avg_elixir
Removing temp directory /tmp/custo_de_elixir.root.20250927.001030.498089...


In [96]:
%%file media_coroas_partida.py
import os
import shutil
from mrjob.job import MRJob
from collections import Counter

class MRCoroas(MRJob):
  def mapper(self, _, line):
    # Ignorar cabeçalho
    if line.startswith("player"):
      return

    colunas = line.strip().split(",")

    try:
      # Coluna 3 = índice 2, Coluna 4 = índice 3
      valor1 = float(colunas[4].strip())
      valor2 = float(colunas[5].strip())
      soma = valor1 + valor2
      yield "soma", soma
    except ValueError:
      pass  # ignora se não for número

  def reducer_init(self):
    self.valores = []

  def reducer(self, key, valores):
    for v in valores:
      self.valores.append(v)

  def reducer_final(self):
    if self.valores:
      media = sum(self.valores) / len(self.valores)
      # primeira linha: média
      yield "Media de coroas por partida:", round(media, 2)

      # depois: contagem de cada soma
      contagem = Counter(self.valores)
      for soma, qtd in sorted(contagem.items()):
        yield f"Total de coroas da partida = {soma}:", f"Numero de partidas = {qtd}"

if __name__ == "__main__":
  if os.path.exists("avg_cororas"):
    shutil.rmtree("avg_coroas")
  MRCoroas.run()


Overwriting media_coroas_partida.py


In [97]:
!python media_coroas_partida.py partidas_clash.csv --output avg_coroas

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 1...
Creating temp directory /tmp/media_coroas_partida.root.20250927.002421.160302
job output is in avg_coroas
Removing temp directory /tmp/media_coroas_partida.root.20250927.002421.160302...


In [101]:
%%file modo_de_jogo.py
import os
import shutil
from mrjob.job import MRJob

class MRModoDeJogo(MRJob):

  def mapper(self, _, line):
    if line.startswith("player"):
      return
    colunas = line.strip().split(",")
    modo = colunas[6].strip()
    yield modo, 1

  def reducer(self, key, valores):
    yield f"{key}:", sum(valores)

if __name__ == "__main__":
  if os.path.exists("avg_elixir"):
    shutil.rmtree("avg_elixir")
  MRModoDeJogo.run()


Overwriting modo_de_jogo.py


In [102]:
!python modo_de_jogo.py partidas_clash.csv --output modo_de_jogo

No configs found; falling back on auto-configuration
No configs specified for inline runner
Running step 1 of 1...
Creating temp directory /tmp/modo_de_jogo.root.20250927.003327.302187
job output is in modo_de_jogo
Removing temp directory /tmp/modo_de_jogo.root.20250927.003327.302187...
